# NKAI Studio — refresh the live Colab GPU worker

Run these cells **top to bottom in the existing Google Colab notebook with a Tesla T4 GPU**. This pulls fresh `main` code into a separate working folder, installs compatible dependencies, launches the worker in a clean Python subprocess (so a stale notebook import cannot override Transformers), opens a Cloudflare quick tunnel, registers it with the NKAI gateway, and tests real chat + TTS. Do not paste secrets into chat or commit them.

**Required Cloudflare Worker secrets:** `AI_REGISTER_SECRET` and `AI_WORKER_TOKEN`. The value entered as `WORKER_API_TOKEN` below must match `AI_WORKER_TOKEN` exactly. Keep the existing app secret `NKAI_AI_CLIENT_SECRET` configured server-side.

In [ ]:
import os, sys, subprocess, pathlib, textwrap, json, time, socket, urllib.request, urllib.error, getpass, threading, base64, io
print('Python:', sys.version.split()[0])
print('GPU check:')
subprocess.run([sys.executable, '-c', 'import torch; print("PyTorch:", torch.__version__); print("CUDA:", torch.cuda.is_available()); print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")'], check=True)


In [ ]:
# Pull the current main-branch worker into a clean folder.
REPO = 'https://github.com/tenzingtinlay-sys/NKAI-Studio.git'
WORKDIR = '/content/NKAI-Studio-fresh'
subprocess.run(['bash','-lc', f'rm -rf {WORKDIR} && git clone --depth 1 --branch main {REPO} {WORKDIR}'], check=True)
os.chdir(WORKDIR + '/gpu-worker')
print(subprocess.check_output(['git','-C',WORKDIR,'log','-1','--format=%h %s'], text=True).strip())
print('Worker file:', pathlib.Path('app.py').resolve())


In [ ]:
# Install compatible packages. The API server will run in a fresh subprocess, not this notebook kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', 'transformers>=4.45,<5', 'accelerate>=1.0,<2', 'fastapi>=0.115,<1', 'uvicorn[standard]>=0.30,<1', 'soundfile>=0.12,<1', 'uroman>=1.3.1,<2'], check=True)
print('Dependency install finished. Existing CUDA PyTorch is intentionally not replaced.')


In [ ]:
# Enter secrets privately. The worker token must equal Cloudflare's AI_WORKER_TOKEN secret.
GATEWAY = 'https://nkai-avatar-gateway.tenzingtinlay.workers.dev'
WORKER_API_TOKEN = getpass.getpass('Worker bearer token (must match Cloudflare AI_WORKER_TOKEN): ').strip()
AI_REGISTER_SECRET = getpass.getpass('Cloudflare AI_REGISTER_SECRET: ').strip()
assert WORKER_API_TOKEN and AI_REGISTER_SECRET, 'Both secrets are required.'
env = os.environ.copy()
env.update({'WORKER_API_TOKEN': WORKER_API_TOKEN, 'MODEL_ID': 'Qwen/Qwen2.5-1.5B-Instruct', 'MAX_NEW_TOKENS': '240', 'CORS_ORIGINS': '*'})
print('Secrets captured in runtime memory only; not written to repository files.')


In [ ]:
# Install Cloudflare quick tunnel client, then launch fresh worker on port 7860.
import platform
cloudflared_path = '/content/cloudflared'
subprocess.run(['bash','-lc', f'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {cloudflared_path} && chmod +x {cloudflared_path}'], check=True)
worker_log = open('/content/nkai-worker.log','w')
worker_proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '7860'], cwd=WORKDIR+'/gpu-worker', env=env, stdout=worker_log, stderr=subprocess.STDOUT, start_new_session=True)
def wait_http(url, timeout=240):
    end=time.time()+timeout; last=None
    while time.time()<end:
        if worker_proc.poll() is not None: raise RuntimeError('Worker process exited. See /content/nkai-worker.log')
        try:
            with urllib.request.urlopen(url, timeout=4) as resp: return json.loads(resp.read().decode())
        except Exception as e: last=e; time.sleep(2)
    raise TimeoutError(f'Worker did not become ready: {last}')
print('Waiting for Qwen model startup; first load can take several minutes...')
print(wait_http('http://127.0.0.1:7860/health', timeout=360))
tunnel_log = open('/content/nkai-cloudflared.log','w')
tunnel_proc = subprocess.Popen([cloudflared_path, 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)
import re
public_url=None; deadline=time.time()+90
while time.time()<deadline and not public_url:
    try: log=pathlib.Path('/content/nkai-cloudflared.log').read_text(errors='ignore')
    except Exception: log=''
    m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
    if m: public_url=m.group(0); break
    if tunnel_proc.poll() is not None: raise RuntimeError('Cloudflare tunnel exited. See /content/nkai-cloudflared.log')
    time.sleep(1)
assert public_url, 'Tunnel URL not found; inspect /content/nkai-cloudflared.log.'
print('Fresh worker tunnel:', public_url)


In [ ]:
# Register the new URL with the AI worker registry and maintain its 90-second lease.
def post_json(url, body, token=None, timeout=30):
    data=json.dumps(body).encode()
    headers={'Content-Type':'application/json'}
    if token: headers['Authorization']='Bearer '+token
    req=urllib.request.Request(url, data=data, headers=headers, method='POST')
    with urllib.request.urlopen(req, timeout=timeout) as resp: return json.loads(resp.read().decode())
def register():
    return post_json(GATEWAY+'/api/ai/register', {'id':'nkai-gpu-worker','url':public_url,'priority':10}, AI_REGISTER_SECRET)
print('Registration:', register())
stop_heartbeat=threading.Event()
def heartbeat_loop():
    while not stop_heartbeat.wait(30):
        try: post_json(GATEWAY+'/api/ai/heartbeat', {'id':'nkai-gpu-worker'}, AI_REGISTER_SECRET)
        except Exception as e: print('Heartbeat warning:', str(e)[:200])
heartbeat_thread=threading.Thread(target=heartbeat_loop, daemon=True); heartbeat_thread.start()
print('Heartbeat running every 30 seconds. Keep this Colab runtime alive.')


In [ ]:
# End-to-end authenticated tests: gateway capabilities, chat, and generated WAV audio.
def request_json(url, method='GET', body=None, token=None, timeout=120):
    data=None if body is None else json.dumps(body).encode()
    headers={}
    if data is not None: headers['Content-Type']='application/json'
    if token: headers['Authorization']='Bearer '+token
    req=urllib.request.Request(url, data=data, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=timeout) as resp: return json.loads(resp.read().decode())
caps=request_json(GATEWAY+'/api/ai/capabilities', token=WORKER_API_TOKEN)
print('Gateway capabilities:', json.dumps(caps, ensure_ascii=False)[:1800])
chat_result=request_json(GATEWAY+'/api/ai/chat', method='POST', body={'message':'Reply with exactly: NKAI fresh worker chat OK','language':'en'}, token=WORKER_API_TOKEN)
print('Chat test:', json.dumps(chat_result, ensure_ascii=False)[:700])
tts_result=request_json(GATEWAY+'/api/ai/tts', method='POST', body={'text':'Hello, this is the NKAI Studio live voice test.','language':'en'}, token=WORKER_API_TOKEN, timeout=180)
audio=base64.b64decode(tts_result.get('audio_base64',''))
assert len(audio)>1000 and audio[:4]==b'RIFF' and audio[8:12]==b'WAVE', 'TTS response did not contain a valid WAV file.'
pathlib.Path('/content/nkai-tts-smoke-test.wav').write_bytes(audio)
print('TTS TEST PASSED:', tts_result.get('model'), tts_result.get('mime_type'), 'bytes=',len(audio), 'saved=/content/nkai-tts-smoke-test.wav')
from IPython.display import Audio, display
display(Audio(audio, rate=None))


## Notes
- If registration returns `401`, the entered registration secret does not match Cloudflare's `AI_REGISTER_SECRET`.
- If gateway requests return `401` or `502`, verify Cloudflare `AI_WORKER_TOKEN` equals the worker token entered above.
- If TTS fails on first request, inspect `/content/nkai-worker.log`; the first language model download can take time. TTS is not confirmed until the final smoke-test cell passes.
- Colab free runtimes can disconnect. A notebook cannot guarantee 24/7 service.
- The default Facebook MMS-TTS checkpoints are generally CC-BY-NC 4.0; verify license rights before commercial use.
